In [2]:
import json

import collections
from ortools.sat.python import cp_model

In [3]:
with open("../data/crop_rollouts.json", "r") as f:
    data = json.load(f)

# print(data)

In [68]:
NUM_DAYS = 30
NUM_TILES = 9

PROFILE = "no_fert"
crops = {k:v[PROFILE] for k,v in data["crops"].items()}

elements = []
for tile in range(NUM_TILES):
    for day in range(NUM_DAYS):
        elements.append((tile, day))

subsets = []
for crop, crop_data in crops.items():
    for el in elements:
        subset = {
            "id": f"S{len(subsets)}",
            "crop": crop,
            "tile": el[0],
            "day": el[1],
            "subset": set(),
            "weight": crop_data["expected_yield"]
        }

        element_exists = True
        for day in crop_data["days"]:
            day_coord = el[1] + day["age"]
            if day_coord < NUM_DAYS:
                subset["subset"].add((el[0], day_coord))
            else:
                element_exists = False
                break

        if element_exists:
            subsets.append(subset)
        else:
            element_exists = True

ops_count = {}
for crop, crop_data in crops.items():
    ops_count.setdefault(crop, {})

    for day in crop_data["days"]:
        ops_count[crop][day["age"]] = len(day["actions"])

In [62]:
# Declare the model
model = cp_model.CpModel()

In [63]:
# Define the variables - one binary decision var per candidate subset
y = {s["id"]: model.NewBoolVar(s["id"]) for s in subsets}

In [65]:
# Packing constraint: each element covered by at most one selected subset
for e in elements:
    covering = [y[s["id"]] for s in subsets if e in s["subset"]]
    if covering:
        model.Add(sum(covering) <= 1)

In [ ]:
# Daily limit on the operations: 16 is the max number of operations per day
for day in range(NUM_DAYS):
    operations = []
    for s in subsets:
        subset_days = {e[1] for e in s["subset"]}
        if day in subset_days:
            age = day - s["day"]
            operations.append(y[s["id"]] * ops_count[s["crop"]][age])
    model.Add(sum(operations) <= 16)

In [71]:
# Objective: maximize the total expected yield
model.Maximize(sum(s["weight"] * y[s["id"]] for s in subsets))

In [78]:
solver = cp_model.CpSolver()
status = solver.Solve(model)

print(status, solver.StatusName(status))
print("objective:", solver.ObjectiveValue())

CpSolverStatus.OPTIMAL OPTIMAL
objective: 212.0


In [79]:
selected = [s for s in subsets if solver.Value(y[s["id"]]) == 1]
print("n selected:", len(selected))

n selected: 54


In [80]:
selected

[{'id': 'S0',
  'crop': 'WHEAT',
  'tile': 0,
  'day': 0,
  'subset': {(0, 0), (0, 1), (0, 2), (0, 3), (0, 4)},
  'weight': 4},
 {'id': 'S5',
  'crop': 'WHEAT',
  'tile': 0,
  'day': 5,
  'subset': {(0, 5), (0, 6), (0, 7), (0, 8), (0, 9)},
  'weight': 4},
 {'id': 'S10',
  'crop': 'WHEAT',
  'tile': 0,
  'day': 10,
  'subset': {(0, 10), (0, 11), (0, 12), (0, 13), (0, 14)},
  'weight': 4},
 {'id': 'S15',
  'crop': 'WHEAT',
  'tile': 0,
  'day': 15,
  'subset': {(0, 15), (0, 16), (0, 17), (0, 18), (0, 19)},
  'weight': 4},
 {'id': 'S20',
  'crop': 'WHEAT',
  'tile': 0,
  'day': 20,
  'subset': {(0, 20), (0, 21), (0, 22), (0, 23), (0, 24)},
  'weight': 4},
 {'id': 'S25',
  'crop': 'WHEAT',
  'tile': 0,
  'day': 25,
  'subset': {(0, 25), (0, 26), (0, 27), (0, 28), (0, 29)},
  'weight': 4},
 {'id': 'S27',
  'crop': 'WHEAT',
  'tile': 1,
  'day': 1,
  'subset': {(1, 1), (1, 2), (1, 3), (1, 4), (1, 5)},
  'weight': 4},
 {'id': 'S32',
  'crop': 'WHEAT',
  'tile': 1,
  'day': 6,
  'subset': {(1,